# T37 - 同花顺Excel导入财务指标

## 项目概述

### 功能描述
本项目专注于从同花顺iFinD API批量提取债券发行人财务指标数据，通过Excel模板生成、数据填充和批量计算，最终将数据导入PostgreSQL数据库并转换格式。

### 核心功能
1. **同花顺API批量提取**：从iFinD API提取150+个债券发行人财务指标
2. **Excel模板处理**：生成模板、填充日期、自动计算
3. **数据导入PostgreSQL**：批量导入数据并进行格式转换
4. **宽表转长表**：使用pandas melt函数转换数据格式

### 数据源
- **同花顺iFinD API**：通过`THS_BD()`函数提取财务指标
- **MySQL数据库**：bond、yq库（债券基础信息）
- **PostgreSQL数据库**：tsdb库（财务指标存储）

### 输出结果
- **指标数据3表**：宽表格式（thscode, dt, 指标1, 指标2, ..., 指标150+）
- **指标数据4表**：长表格式（dt, trade_code, 指标, value）

---

## 1. 环境配置

### 1.1 导入依赖库

In [ ]:
# 标准库
import os
import time
import logging
from time import sleep
from datetime import datetime, date, timedelta
from pathlib import Path

# 数据处理
import numpy as np
import pandas as pd

# 数据库
import sqlalchemy
from sqlalchemy import text, create_engine
from sqlalchemy.pool import NullPool
import psycopg2

# Excel操作
from openpyxl import load_workbook
import win32com.client as win32

# 同花顺iFinD（需要安装iFinDPy库）
try:
    from iFinDPy import THS_BD, THS_iFinDLogin
    IFIND_AVAILABLE = True
except ImportError:
    IFIND_AVAILABLE = False
    print("警告：iFinDPy库未安装，同花顺API功能不可用")

# 配置
from config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR,
    DB_HOST, DB_PORT, DB_USER, DB_PASSWORD, DB_NAME,
    PG_HOST, PG_PORT, PG_USER, PG_PASSWORD, PG_DB,
    IFIND_USERNAME, IFIND_PASSWORD,
    CHUNKSIZE
)

print("环境配置完成")

### 1.2 日志配置

In [ ]:
# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(PROJECT_ROOT / 'process.log', encoding='utf-8')
    ]
)
logger = logging.getLogger(__name__)
logger.info("日志系统初始化完成")

---

## 2. 数据获取

### 2.1 数据库连接管理

In [ ]:
class DatabaseManager:
    """数据库连接管理器，支持MySQL和PostgreSQL"""
    
    def __init__(self, db_type='mysql', host=None, port=None, user=None, password=None, database=None):
        self.db_type = db_type
        self.host = host
        self.port = port
        self.user = user
        self.password = password
        self.database = database
        self.engine = None
        self.connection = None
        self.cursor = None
        self._connect()
    
    def _connect(self):
        """建立数据库连接"""
        max_retries = 3
        retry_count = 0
        
        while retry_count < max_retries:
            try:
                if self.db_type == 'mysql':
                    self.engine = create_engine(
                        f'mysql+pymysql://{self.user}:{self.password}@{self.host}:{self.port}/{self.database}',
                        poolclass=NullPool,
                        pool_recycle=3600
                    )
                    self.connection = self.engine.connect()
                elif self.db_type == 'postgresql':
                    self.engine = create_engine(
                        f'postgresql://{self.user}:{self.password}@{self.host}:{self.port}/{self.database}'
                    )
                    self.connection = self.engine.connect()
                    # 同时创建psycopg2连接用于DDL操作
                    self.pg_conn = psycopg2.connect(
                        host=self.host,
                        port=self.port,
                        user=self.user,
                        password=self.password,
                        database=self.database
                    )
                    self.cursor = self.pg_conn.cursor()
                
                logger.info(f"{self.db_type}数据库连接成功: {self.host}:{self.port}/{self.database}")
                return
                
            except Exception as e:
                logger.error(f"数据库连接失败 (尝试 {retry_count + 1}/{max_retries}): {e}")
                retry_count += 1
                sleep(1)
        
        raise Exception(f"无法连接到数据库: {self.host}:{self.port}/{self.database}")
    
    def execute_sql(self, sql, params=None):
        """执行SQL语句（带重试机制）"""
        max_retries = 3
        retry_count = 0
        
        while retry_count < max_retries:
            try:
                if self.db_type == 'postgresql' and self.cursor:
                    self.cursor.execute(sql, params)
                    self.pg_conn.commit()
                else:
                    trans = self.connection.begin()
                    try:
                        self.connection.execute(text(sql), params)
                        trans.commit()
                    except Exception as e:
                        trans.rollback()
                        raise e
                return
            except Exception as e:
                logger.error(f"SQL执行失败 (尝试 {retry_count + 1}/{max_retries}): {e}")
                retry_count += 1
                self._reconnect()
                sleep(1)
        
        raise Exception(f"SQL执行失败: {sql[:100]}...")
    
    def _reconnect(self):
        """重新建立连接"""
        try:
            if self.connection:
                self.connection.close()
        except:
            pass
        self._connect()
    
    def read_sql(self, sql):
        """读取SQL查询结果为DataFrame"""
        return pd.read_sql(sql, self.connection)
    
    def to_sql(self, df, table_name, if_exists='append'):
        """将DataFrame写入数据库表"""
        df.to_sql(table_name, self.connection, if_exists=if_exists, index=False)
    
    def close(self):
        """关闭数据库连接"""
        try:
            if self.connection:
                self.connection.close()
            if self.cursor:
                self.cursor.close()
            if hasattr(self, 'pg_conn'):
                self.pg_conn.close()
            logger.info("数据库连接已关闭")
        except Exception as e:
            logger.error(f"关闭连接时出错: {e}")

print("DatabaseManager类定义完成")

### 2.2 初始化数据库连接

In [ ]:
# MySQL bond库连接（债券基础信息）
db_bond = DatabaseManager(
    db_type='mysql',
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

# PostgreSQL tsdb库连接（财务指标存储）
db_tsdb = DatabaseManager(
    db_type='postgresql',
    host=PG_HOST,
    port=PG_PORT,
    user=PG_USER,
    password=PG_PASSWORD,
    database=PG_DB
)

print("数据库连接初始化完成")

### 2.3 同花顺iFinD API登录

In [ ]:
def login_ifind(username=None, password=None):
    """登录同花顺iFinD API"""
    if not IFIND_AVAILABLE:
        logger.warning("iFinDPy库未安装，跳过登录")
        return False
    
    username = username or IFIND_USERNAME
    password = password or IFIND_PASSWORD
    
    if not username or not password:
        logger.warning("未配置iFinD账号密码，跳过登录")
        return False
    
    try:
        result = THS_iFinDLogin(username, password)
        if result == 0:
            logger.info("iFinD API登录成功")
            return True
        else:
            logger.error(f"iFinD API登录失败，错误码: {result}")
            return False
    except Exception as e:
        logger.error(f"iFinD API登录异常: {e}")
        return False

# 登录iFinD
ifind_logged_in = login_ifind()

---

## 3. 数据处理

### 3.1 获取债券发行人代码

In [ ]:
def get_issuer_trade_codes(db_manager):
    """
    从数据库获取所有债券发行人代码（去重）
    
    逻辑：
    - 从三个基础表（信用债/金融债/ABS）提取发行人代码和名称
    - 按发行人分组，取最新的trade_code
    
    Returns:
        list: trade_code列表
    """
    sql = """
    SELECT 
        max(trade_code) AS trade_code,
        ths_issuer_name_cn_bond
    FROM (
        SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_credit
        UNION ALL
        SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_finance
        UNION ALL
        SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_abs
    ) SQ
    GROUP BY ths_issuer_name_cn_bond
    """
    
    df = db_manager.read_sql(sql)
    trade_codes = df['trade_code'].tolist()
    logger.info(f"获取到 {len(trade_codes)} 个发行人代码")
    return trade_codes

# 获取发行人代码
trade_codes = get_issuer_trade_codes(db_tsdb)
print(f"发行人数量: {len(trade_codes)}")

### 3.2 财务指标定义

定义150+个财务指标常量

In [ ]:
# 财务指标列表（150+个）
FINANCIAL_INDICATORS = [
    # 资产负债表指标
    'ths_total_assets_bond',                    # 总资产
    'ths_cash_in_inventory_detail_bond',        # 库存现金
    'ths_bank_deposit_detail_bond',             # 银行存款
    'ths_other_currency_fund_detail_bond',      # 其他货币资金
    'ths_currency_fund_detail_sum_bond',        # 货币资金合计
    'ths_account_receivable_bond',              # 应收账款
    'ths_bills_receivable_bond',                # 应收票据
    'ths_bill_and_account_receivable_bond',     # 应收票据及应收账款
    'ths_receivable_financing_bond',            # 应收款项融资
    'ths_other_receivables_bond',               # 其他应收款
    'ths_inventory_bond',                       # 存货
    'ths_contract_asset_bond',                  # 合同资产
    'ths_total_current_assets_bond',            # 流动资产合计
    'ths_fixed_asset_bond',                     # 固定资产
    'ths_construction_in_process_bond',         # 在建工程
    'ths_right_of_use_assets_bond',             # 使用权资产
    'ths_st_borrow_bond',                       # 短期借款
    'ths_bill_payable_bond',                    # 应付票据
    'ths_accounts_payable_bond',                # 应付账款
    'ths_contract_liab_bond',                   # 合同负债
    'ths_payroll_payable_bond',                 # 应付职工薪酬
    'ths_tax_payable_bond',                     # 应交税费
    'ths_interest_payable_bond',                # 应付利息
    'ths_dividend_payable_bond',                # 应付股利
    'ths_total_current_liab_bond',              # 流动负债合计
    'ths_lt_loan_bond',                         # 长期借款
    'ths_bond_payable_bond',                    # 应付债券
    'ths_lease_libilities_bond',                # 租赁负债
    'ths_total_noncurrent_liab_bond',           # 非流动负债合计
    'ths_total_liab_bond',                      # 负债合计
    'ths_actual_received_capital_bond',         # 实收资本
    'ths_capital_reserve_bond',                 # 资本公积
    'ths_undstrbtd_profit_bond',                # 未分配利润
    'ths_total_owner_equity_bond',              # 所有者权益合计
    
    # 利润表指标
    'ths_ebit_bond',                            # 息税前利润
    'ths_ebitda_bond',                          # 息税折旧摊销前利润
    'ths_continue_operate_net_profit_ttm_bond', # 持续经营净利润TTM
    'ths_ebitda_ttm_bond',                      # EBITDA TTM
    'ths_ebit_ttm_bond',                        # EBIT TTM
    
    # 现金流量表指标
    'ths_cash_received_of_borrowing_bond',      # 借款收到的现金
    'ths_cash_received_from_bond_issue_bond',   # 发行债券收到的现金
    'ths_invest_income_cash_received_bond',     # 投资收益收到的现金
    'ths_cash_received_of_sales_service_bond',  # 销售收到的现金
    
    # 财务比率指标
    'ths_annaul_net_asset_yield_bond',          # 年净资产收益率
    'ths_gross_selling_rate_bond',              # 销售毛利率
    'ths_roe_avg_by_ths_bond',                  # ROE
    'ths_asset_liab_ratio_bond',                # 资产负债率
]

# 完整指标字符串（用于API调用）
INDICATORS_STR = ';'.join(FINANCIAL_INDICATORS)

print(f"定义了 {len(FINANCIAL_INDICATORS)} 个财务指标")

---

## 4. 核心逻辑

### 4.1 同花顺API批量提取财务指标

In [ ]:
def process_trade_code(trade_code, dt, dt1, indicators_str):
    """
    使用同花顺API提取单个发行人的财务指标
    
    Args:
        trade_code: 债券代码（代表发行人）
        dt: 报告期（季度）格式：'20230630'
        dt1: 上期（年度）格式：'2022-12-31'
        indicators_str: 指标字符串，用分号分隔
    
    Returns:
        DataFrame: 财务指标数据
    """
    if not IFIND_AVAILABLE:
        logger.warning("iFinD API不可用")
        return pd.DataFrame()
    
    try:
        # 构建日期参数
        # 参数格式：'{指标1},{频率};{指标2},{频率};...'
        # 频率100 = 季度数据，无频率 = 年度数据
        date_params = f'{dt},100;{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},5;{dt},5;{dt},5;{dt},5;{dt},5;{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt};{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt};{dt};{dt1},100;{dt1},100;{dt1},100;{dt1},100;{dt1},100;{dt},100;{dt},100;{dt},100;{dt};{dt};{dt};{dt};{dt};{dt};{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt},100;{dt};{dt};{dt};{dt};{dt};{dt}'
        
        # 调用API
        df = THS_BD(trade_code, indicators_str, date_params)
        df = df.data
        
        if df is None:
            return pd.DataFrame()
        
        df['dt'] = dt
        return df
        
    except Exception as e:
        logger.error(f"处理 {trade_code} 时出错: {e}")
        return pd.DataFrame()


def batch_extract_financial_indicators(trade_codes, dates, indicators_str, db_manager, table_name='指标数据3'):
    """
    批量提取财务指标并写入数据库
    
    Args:
        trade_codes: 发行代码列表
        dates: 日期列表 [(dt, dt1), ...]
        indicators_str: 指标字符串
        db_manager: 数据库管理器
        table_name: 目标表名
    
    Returns:
        int: 成功处理的记录数
    """
    processed_count = 0
    failed_count = 0
    
    for dt, dt1 in dates:
        logger.info(f"开始处理日期: {dt}")
        
        for i, trade_code in enumerate(trade_codes):
            try:
                df = process_trade_code(trade_code, dt, dt1, indicators_str)
                
                if df.empty:
                    continue
                
                df.dropna(how='all', inplace=True)
                
                if not df.empty:
                    db_manager.to_sql(df, table_name, if_exists='append')
                    processed_count += len(df)
                
                # 进度显示
                if (i + 1) % 100 == 0:
                    logger.info(f"已处理 {i + 1}/{len(trade_codes)} 个发行人")
                    
            except Exception as e:
                logger.error(f"处理 {trade_code} ({dt}) 失败: {e}")
                failed_count += 1
    
    logger.info(f"批量提取完成: 成功 {processed_count} 条, 失败 {failed_count} 个")
    return processed_count

print("API提取函数定义完成")

### 4.2 Excel批量处理模块

In [ ]:
class ExcelProcessor:
    """Excel批量处理类"""
    
    def __init__(self):
        self.excel_app = None
    
    def _get_excel_app(self):
        """获取Excel应用实例"""
        if self.excel_app is None:
            self.excel_app = win32.gencache.EnsureDispatch('Excel.Application')
        return self.excel_app
    
    def fill_dates_from_filename(self, folder_path):
        """
        从文件名提取日期并填充到Excel最右列
        
        Args:
            folder_path: Excel文件所在文件夹路径
        """
        folder_path = Path(folder_path)
        files = [f for f in folder_path.iterdir() if f.suffix == '.xlsx']
        
        excel = self._get_excel_app()
        
        for file_path in files:
            try:
                wb = excel.Workbooks.Open(str(file_path))
                ws = wb.Worksheets(1)
                
                # 从文件名提取日期（格式：YYYYMMDD.xlsx）
                date_str = file_path.stem  # '20220630'
                date_obj = datetime.strptime(date_str, '%Y%m%d')
                formatted_date = date_obj.strftime('%Y-%m-%d')  # '2022-06-30'
                
                # 获取最右列
                last_column = ws.Cells(1, ws.Columns.Count).End(win32.constants.xlToLeft).Column
                
                # 填充日期值
                ws.Range(
                    ws.Cells(2, last_column),
                    ws.Cells(ws.UsedRange.Rows.Count, last_column)
                ).Value = formatted_date
                
                # 自动填充
                col = last_column - 1
                source_range = ws.Range(ws.Cells(2, col), ws.Cells(2, col))
                destination_range = ws.Range(ws.Cells(2, col), ws.Cells(ws.UsedRange.Rows.Count, col))
                source_range.AutoFill(destination_range, win32.constants.xlFillDefault)
                
                wb.Save()
                wb.Close()
                logger.info(f"已处理: {file_path.name}")
                
            except Exception as e:
                logger.error(f"处理 {file_path.name} 时出错: {e}")
        
        logger.info(f"日期填充完成，共处理 {len(files)} 个文件")
    
    def calculate_and_save_as_values(self, folder_path):
        """
        批量执行Excel计算并保存为值
        
        Args:
            folder_path: Excel文件所在文件夹路径
        """
        folder_path = Path(folder_path)
        files = [f for f in folder_path.iterdir() if f.suffix == '.xlsx']
        
        excel = self._get_excel_app()
        
        for file_path in files:
            try:
                wb = excel.Workbooks.Open(str(file_path))
                ws = wb.Worksheets(1)
                
                # 等待计算完成
                sleep(60)
                ws.Calculate()
                
                # 将公式结果保存为值
                used_range = ws.UsedRange
                used_range.Copy()
                used_range.PasteSpecial(Paste=12)  # xlPasteValues
                ws.Application.CutCopyMode = False
                
                wb.Save()
                wb.Close()
                logger.info(f"已计算并保存: {file_path.name}")
                
            except Exception as e:
                logger.error(f"处理 {file_path.name} 时出错: {e}")
        
        logger.info(f"计算保存完成，共处理 {len(files)} 个文件")
    
    def quit(self):
        """关闭Excel应用"""
        if self.excel_app:
            self.excel_app.Quit()
            self.excel_app = None

print("ExcelProcessor类定义完成")

### 4.3 数据库导入与格式转换

In [ ]:
def import_excel_to_postgres(folder_path, db_manager, table_name='指标数据3'):
    """
    批量将Excel文件导入PostgreSQL
    
    Args:
        folder_path: Excel文件所在文件夹路径
        db_manager: PostgreSQL数据库管理器
        table_name: 目标表名
    """
    folder_path = Path(folder_path)
    files = [f for f in folder_path.iterdir() if f.suffix in ['.xlsx', '.xls']]
    
    imported_count = 0
    
    for file_path in files:
        try:
            df = pd.read_excel(file_path)
            db_manager.to_sql(df, table_name, if_exists='append')
            imported_count += len(df)
            logger.info(f"已导入: {file_path.name}, {len(df)} 行")
        except Exception as e:
            logger.error(f"导入 {file_path.name} 时出错: {e}")
    
    logger.info(f"导入完成，共 {imported_count} 行")
    return imported_count


def convert_columns_to_float(db_manager, table_name, exclude_columns=None):
    """
    将表中除指定列外的所有列转换为FLOAT类型
    
    Args:
        db_manager: PostgreSQL数据库管理器
        table_name: 表名
        exclude_columns: 排除的列名列表
    """
    exclude_columns = exclude_columns or ['dt', 'thscode']
    
    # 获取所有列名
    sql = f"""
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_name = '{table_name}'
    """
    df = db_manager.read_sql(sql)
    columns = df['column_name'].tolist()
    
    # 排除不需要转换的列
    columns_to_alter = [col for col in columns if col not in exclude_columns]
    
    # 批量转换
    for col in columns_to_alter:
        sql = f"""
        ALTER TABLE {table_name}
        ALTER COLUMN {col} TYPE FLOAT USING {col}::FLOAT
        """
        db_manager.execute_sql(sql)
    
    logger.info(f"已转换 {len(columns_to_alter)} 列为FLOAT类型")


def wide_to_long_table(db_manager, source_table, target_table, id_vars=None, chunksize=10000):
    """
    将宽表转换为长表
    
    Args:
        db_manager: 数据库管理器
        source_table: 源表名（宽表）
        target_table: 目标表名（长表）
        id_vars: 保持不变的列
        chunksize: 分块大小
    """
    id_vars = id_vars or ['dt', 'thscode']
    
    query = f"SELECT * FROM {source_table}"
    chunks = pd.read_sql(query, db_manager.connection, chunksize=chunksize)
    
    processed_rows = 0
    
    for df in chunks:
        # 宽表转长表
        df = df.melt(
            id_vars=id_vars,
            var_name='指标',
            value_name='value'
        )
        
        # 重命名
        df.rename(columns={'thscode': 'trade_code'}, inplace=True)
        
        # 写入新表
        try:
            df.to_sql(target_table, db_manager.connection, if_exists='append', index=False)
        except Exception as e:
            logger.error(f"写入 {target_table} 时出错: {e}")
        
        processed_rows += len(df)
        logger.info(f"已处理 {processed_rows} 行")
    
    logger.info(f"宽表转长表完成，共 {processed_rows} 行")
    return processed_rows

print("数据库处理函数定义完成")

---

## 5. 执行与测试

### 5.1 执行主流程

In [ ]:
def main():
    """主执行函数"""
    logger.info("="*50)
    logger.info("开始执行 T37-同花顺Excel导入财务指标")
    logger.info("="*50)
    
    # 步骤1：获取发行人代码
    logger.info("步骤1：获取发行人代码")
    trade_codes = get_issuer_trade_codes(db_tsdb)
    
    # 步骤2：定义提取日期
    dates = [
        ('20230630', '2023-06-30'),
        ('20221231', '2022-12-31')
    ]
    
    # 步骤3：批量提取财务指标（需要iFinD API）
    if ifind_logged_in:
        logger.info("步骤2：批量提取财务指标")
        count = batch_extract_financial_indicators(
            trade_codes, dates, INDICATORS_STR, db_tsdb, '指标数据3'
        )
        logger.info(f"提取完成，共 {count} 条记录")
    else:
        logger.warning("iFinD API未登录，跳过提取步骤")
    
    # 步骤4：列类型转换
    logger.info("步骤3：列类型转换")
    convert_columns_to_float(db_tsdb, '指标数据3')
    
    # 步骤5：宽表转长表
    logger.info("步骤4：宽表转长表转换")
    count = wide_to_long_table(db_tsdb, '指标数据3', '指标数据4', chunksize=CHUNKSIZE)
    
    logger.info("="*50)
    logger.info("T37执行完成")
    logger.info("="*50)

# 执行主流程
if __name__ == '__main__':
    main()

### 5.2 Excel处理执行示例

In [ ]:
def process_excel_files(folder_path):
    """
    Excel文件处理示例
    
    Args:
        folder_path: Excel文件所在文件夹
    """
    processor = ExcelProcessor()
    
    try:
        # 步骤1：填充日期
        logger.info("步骤1：填充日期列")
        processor.fill_dates_from_filename(folder_path)
        
        # 步骤2：计算并保存为值
        logger.info("步骤2：计算并保存为值")
        processor.calculate_and_save_as_values(folder_path)
        
        # 步骤3：导入数据库
        logger.info("步骤3：导入数据库")
        import_excel_to_postgres(folder_path, db_tsdb, '指标数据3')
        
    finally:
        processor.quit()

# 示例调用（需要实际路径）
# process_excel_files(r'C:\Users\Administrator\Desktop\新建文件夹')

---

## 6. 工具函数（可复用）

### 6.1 重试装饰器

In [ ]:
def retry(attempts=3, delay=30):
    """
    重试装饰器
    
    Args:
        attempts: 最大重试次数
        delay: 重试间隔（秒）
    """
    def decorator(func):
        def wrapper(*args, **kwargs):
            for i in range(attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    logger.warning(f"尝试 {i+1}/{attempts} 失败: {e}, {delay}秒后重试...")
                    time.sleep(delay)
            raise Exception(f"所有 {attempts} 次尝试均失败")
        return wrapper
    return decorator

print("重试装饰器定义完成")

### 6.2 日期处理工具

In [ ]:
def parse_date_from_filename(filename, pattern='%Y%m%d'):
    """
    从文件名中解析日期
    
    Args:
        filename: 文件名（不含扩展名）
        pattern: 日期格式
    
    Returns:
        str: 格式化的日期字符串 (YYYY-MM-DD)
    """
    date_obj = datetime.strptime(filename, pattern)
    return date_obj.strftime('%Y-%m-%d')


def generate_quarter_dates(start_year=2020, end_year=2023):
    """
    生成季度报告日期列表
    
    Returns:
        list: [(dt, dt1), ...] 格式的日期列表
    """
    dates = []
    for year in range(start_year, end_year + 1):
        quarters = [
            (f'{year}0331', f'{year-1}-12-31'),
            (f'{year}0630', f'{year}-03-31'),
            (f'{year}0930', f'{year}-06-30'),
            (f'{year}1231', f'{year}-09-30')
        ]
        dates.extend(quarters)
    return dates

# 示例
quarter_dates = generate_quarter_dates(2022, 2023)
print(f"生成 {len(quarter_dates)} 个季度日期")
print(quarter_dates)

### 6.3 数据验证工具

In [ ]:
def validate_financial_data(df, required_columns=None):
    """
    验证财务数据完整性
    
    Args:
        df: 待验证的DataFrame
        required_columns: 必需的列名列表
    
    Returns:
        dict: 验证结果
    """
    required_columns = required_columns or ['dt', 'thscode']
    
    result = {
        'is_valid': True,
        'missing_columns': [],
        'null_counts': {},
        'row_count': len(df)
    }
    
    # 检查必需列
    for col in required_columns:
        if col not in df.columns:
            result['missing_columns'].append(col)
            result['is_valid'] = False
    
    # 检查空值
    for col in df.columns:
        null_count = df[col].isnull().sum()
        if null_count > 0:
            result['null_counts'][col] = null_count
    
    return result

print("数据验证工具定义完成")

---

## 附录：清理资源

In [ ]:
def cleanup():
    """清理资源，关闭连接"""
    try:
        db_bond.close()
        db_tsdb.close()
        logger.info("资源清理完成")
    except Exception as e:
        logger.error(f"清理资源时出错: {e}")

# cleanup()